# News Clustering (Basic)

Notebook ini memperkenalkan clustering berita menggunakan data Excel Anda.

## Sumber data
- File: 08-Text Mining/news_clf.xlsx
- Sheet: training

## Target belajar
1. Membaca data teks dari Excel
2. Membersihkan teks sederhana
3. Mengubah teks menjadi angka (TF-IDF)
4. Melakukan clustering dengan KMeans
5. Membaca hasil cluster

## 1) Install dan import library

In [ ]:
# Uncomment jika library belum ada
# !pip install pandas openpyxl scikit-learn

In [ ]:
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

## 2) Baca data dari Excel (sheet training)

In [ ]:
path_file = "news_clf.xlsx"
sheet_name = "training"

try:
    df = pd.read_excel(path_file, sheet_name=sheet_name)
except ValueError:
    xls = pd.ExcelFile(path_file)
    print("Sheet tidak ditemukan. Daftar sheet tersedia:", xls.sheet_names)
    raise

print("Ukuran data:", df.shape)
df.head()

## 3) Pilih kolom teks berita
Cell ini mencoba mencari kolom teks otomatis. Jika tidak cocok, ganti manual variabel text_col.

In [ ]:
kandidat_text = ["text", "news", "judul", "isi", "content", "berita"]
lower_map = {c.lower(): c for c in df.columns}

text_col = None
for k in kandidat_text:
    if k in lower_map:
        text_col = lower_map[k]
        break

if text_col is None:
    print("Kolom teks belum terdeteksi otomatis.")
    print("Daftar kolom:", df.columns.tolist())
    # Contoh manual: text_col = "nama_kolom_teks"
else:
    print("Kolom teks terpilih:", text_col)

## 4) Cleaning teks sederhana
Tahap dasar: lowercase, hapus angka/simbol, rapikan spasi.

In [ ]:
def clean_text(s):
    s = str(s).lower()
    s = re.sub(r"[^a-zA-Z\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

df_work = df.copy()
df_work = df_work.dropna(subset=[text_col])
df_work["clean_text"] = df_work[text_col].apply(clean_text)

df_work[[text_col, "clean_text"]].head()

## 5) Ubah teks ke TF-IDF
TF-IDF mengubah teks menjadi fitur numerik agar bisa diproses algoritma ML.

In [ ]:
vectorizer = TfidfVectorizer(max_features=1000, stop_words="english")
X = vectorizer.fit_transform(df_work["clean_text"])
print("Shape TF-IDF:", X.shape)

## 6) Coba beberapa jumlah cluster (k)
Kita pakai silhouette score untuk gambaran kualitas cluster.

In [ ]:
hasil_k = []
k_values = [2, 3, 4, 5]

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    score = silhouette_score(X, labels)
    hasil_k.append((k, score))

pd.DataFrame(hasil_k, columns=["k", "silhouette_score"])

## 7) Clustering final
Untuk kelas basic, kita set k = 3. Boleh diubah sesuai hasil evaluasi di atas.

In [ ]:
k_final = 3
model = KMeans(n_clusters=k_final, random_state=42, n_init=10)
df_work["cluster"] = model.fit_predict(X)

print(df_work["cluster"].value_counts().sort_index())
df_work[[text_col, "cluster"]].head(10)

## 8) Lihat kata kunci utama tiap cluster
Kata dengan bobot tertinggi membantu interpretasi tema cluster.

In [ ]:
terms = vectorizer.get_feature_names_out()
centers = model.cluster_centers_

for i in range(k_final):
    top_idx = centers[i].argsort()[-10:][::-1]
    top_words = [terms[j] for j in top_idx]
    print(f"Cluster {i}:", ", ".join(top_words))

## 9) Simpan hasil clustering

In [ ]:
output_file = "08-Text Mining/news_clustering_result.csv"
df_work.to_csv(output_file, index=False)
print("Hasil tersimpan di:", output_file)